# Challenge 1: The banknote-authentication data set problem

We will perform a nearly realistic analysis of the data set bank note authentication that can be downloaded from this [link](https://archive.ics.uci.edu/dataset/267/banknote+authentication) or from the [github repo](https://github.com/LorenzoTonet/Intro_to_ML_26-27/blob/main/data/data_banknote_authentication.txt)

## Data set description

Data were extracted from images that were taken from genuine and forged banknote-like specimens. For digitization, an industrial camera usually used for print inspection was used. The final images have 400x 400 pixels. Due to the object lens and distance to the investigated object gray-scale pictures with a resolution of about 660 dpi were gained. Wavelet Transform tool were used to extract features from images.
These features are:
1. variance of Wavelet Transformed image (continuous) 
2. skewness of Wavelet Transformed image (continuous) 
3. curtosis of Wavelet Transformed image (continuous) 
4. entropy of image (continuous) 
5. class (integer)

## Task description
We have a binary classification problem. The assignment can be divided in several parts:
    
    1. Load the data and pretreatment.
    2. Data exploring by Unsupervised Learning techniques.
    3. Construction of several models of Supervised Learning.
    4. Assessment and evaluation

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as sk

### 1. Data pretreatment

Load the data and look at it: It is needed some kind of scaling? Why? Are the data points sorted in the original data set? Can it generate problems? How can this be solved?

In [ ]:
path_csv = "../data/data_banknote_authentication.csv"
column_names = ["variance", "skewness", "curtosis", "entropy", "class"]
df = pd.read_csv(path_csv, header=None, names=column_names)

df.head()


In [ ]:
feature_columns = column_names[:-1]

assert df.shape[1] == len(column_names)
assert df[feature_columns].notna().all().all()
assert df["class"].isin([0, 1]).all()

print(f"Shape: {df.shape}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print("Class distribution:")
print(df["class"].value_counts().sort_index())
print("Feature summary:")
display(df[feature_columns].describe().T)


In [ ]:
features = df[feature_columns]
q1 = features.quantile(0.25)
q3 = features.quantile(0.75)
iqr = q3 - q1
outlier_mask = features.lt(q1 - 1.5 * iqr) | features.gt(q3 + 1.5 * iqr)

print("Potential outliers by feature:")
print(outlier_mask.sum())
print(f"Rows with at least one potential outlier: {outlier_mask.any(axis=1).sum()}")


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer

# The target is kept unchanged; only the skewed input features are transformed.
preprocessor = ColumnTransformer(
    [("power", PowerTransformer(method="yeo-johnson"), ["curtosis", "entropy"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
)
X = pd.DataFrame(
    preprocessor.fit_transform(features),
    columns=feature_columns,
    index=df.index,
)
y = df["class"].copy()

print("Skewness before transformation:")
print(features.skew().round(3))
print("Skewness after transformation:")
print(X.skew().round(3))


### 2. Unsupervised Learning

Use PCA and plot the two first components colouring according with the class. Are the classes linearly separable in this projection? What happens when I applied k-means with two classes in this space? And if I use all the coordinates? Try also t-SNE for projection and DBSCAN for the clustering and comment on the results.

### 3. Supervised Learning

Generate a subset of the data of 372 elements that would be saved as test set. With the rest of the data generate the following models: Logistic Regression, Decision tree (use the ID3 algorithm), Naive Bayes and k-NN. 

Investigate the effect of regularization (when possible) and use cross validation for setting the hyper-parameters when needed. 

Compare the performances in terms of accuracy, precision, recall and F1-score on the test set. Comment these results at the light of those obtained from the Unsupervised Learning analysis. Could you propose a way to improve these results?     
